In [ ]:
import argparse
import os, sys
import time
import datetime

# Import pytorch dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from tqdm import tqdm_notebook as tqdm

from dataloader import CIFAR10

In [ ]:
"""
Memory and Computational Cost Analysis

Objective: Calculate the theoretical hardware requirements for the LeNet-5 architecture.
By manually computing the parameter count (memory consumption) and
Multiply-Accumulate operations (computational cost), we gain insight into
the model's footprint and efficiency before deploying it to hardware.
"""

"""Memory Consumption (Number of Parameters)"""
# Layer 1: Conv1
conv1 = (5 * 5 * 3 * 6) + 6
# Layer 2: Conv1
conv2 = (5 * 5 * 6 * 16) + 16
# FC1
FC1 = (400 * 120) + 120
# FC2
FC2 = (120 * 84) + 84
# FC3
FC3 = (84 * 10) + 10
# Total Amount of Parameters: conv1 + conv2 + FC1 + FC2 + FC3
mem_consumption = conv1 + conv2 + FC1 + FC2 + FC3
print(f"The total memory consumption (i.e., number of parameters): {mem_consumption}")

The total memory consumption (i.e., number of parameters): 62006


In [ ]:
"""Computational Cost (MACs)"""
# Layer 1: Conv1
conv1 = (28 * 28 * 6) * (5 * 5 * 3)
# Layer 2: Conv2
conv2 = (10 * 10 * 16) * (5 * 5 * 6)
# FC1
FC1 = 400 * 120
# FC2
FC2 = 120 * 84
# FC3
FC3 = 84 * 10
# Total MACs: conv1 + conv2 + FC1 + FC2 + FC3
total_MACs = conv1 + conv2 + FC1 + FC2 + FC3
print(f"The total computational cost (i.e., MACs): {total_MACs}")

The total computational cost (i.e., MACs): 651720


In [ ]:
"""
Model Architecture Definition

Objective: Implement the LeNet-5 Convolutional Neural Network in PyTorch.
This defines the structural flow of data through convolutional, pooling,
and fully connected layers. We also include an optional Batch Normalization
flag to study its effect on training stability and convergence speed.
"""
# Create the neural network module: LeNet-5
class LeNet5(nn.Module):
    def __init__(self, use_bn: bool = False):
        super(LeNet5, self).__init__()
        self.use_bn = use_bn
        # CIFAR-10 is 32x32 "colorful," which means 3 input channels
        # Convolution
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, stride=1, padding=0)
        # MaxPool
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Convolution
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding=0)
        # After conv/pool: 32 -> 28 -> 14 -> 10 -> 5, channels=16 -> 16 * 5 * 5 = 400
        # Fully Connected (1)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        # Fully Connected (2)
        self.fc2 = nn.Linear(120, 84)
        # Fully Connected (3) -> Output (10 classes)
        self.fc3 = nn.Linear(84, 10)

        # Optional BatchNorm Layers
        if self.use_bn:
          self.bn1_conv1 = nn.BatchNorm2d(6)
          self.bn2_conv2 = nn.BatchNorm2d(16)
          self.bn_fc1 = nn.BatchNorm1d(120)
          self.bn_fc2 = nn.BatchNorm1d(84)


    def forward(self, x):
        # L1 & L2: Conv -> ReLU -> Pool
        x = self.conv1(x)
        if self.use_bn:
          x = self.bn1_conv1(x)
        x = F.relu(x)
        x = self.pool(x) # (N,6,14,14)

        # L3 & L4: Conv -> ReLU -> Pool
        x = self.conv2(x)
        if self.use_bn:
          x = self.bn2_conv2(x)
        x = F.relu(x)
        x = self.pool(x) # (N,16,5,5)

        # Flattening
        x = torch.flatten(x, 1) # (N,400)

        # L5: FC1 -> ReLU
        x = self.fc1(x)
        if self.use_bn:
            x = self.bn_fc1(x)
        x = F.relu(x) # (N,120)

        # L6: FC2 -> ReLU
        x = self.fc2(x)
        if self.use_bn:
            x = self.bn_fc2(x)
        x = F.relu(x) # (N,84)

        # L7: FC3 -> Output
        x = self.fc3(x) # (N,10)

        return x


In [ ]:
"""
Hyperparameter Configuration

Objective: Define the core training hyperparameters.
These settings (batch size, learning rate, momentum, L2 regularization,
and epochs) control the optimization process. Tuning these is critical
for achieving high validation accuracy and preventing overfitting.
"""

# Setting some hyperparameters
TRAIN_BATCH_SIZE = 128
VAL_BATCH_SIZE = 100
INITIAL_LR = 0.01
MOMENTUM = 0.9
REG = 1e-3 # 1e-4
EPOCHS = 40 # 30
DATAROOT = "./data"
CHECKPOINT_PATH = "./saved_model"

In [ ]:
"""
Data Preprocessing and Augmentation

Objective: Prepare the CIFAR-10 image data for neural network training.
This pipeline normalizes the inputs using dataset-specific mean and standard
deviation for faster convergence. It also applies data augmentation
(Random Crop, Random Horizontal Flip) to artificially expand the training
dataset, helping the model generalize better to unseen data.
"""
# Specify preprocessing function.
# Reference mean/std value for
mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)


transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), # Augmentation: Padding + Crop
    transforms.RandomHorizontalFlip(),    # Augmentation: Flip
    transforms.ToTensor(),                # Convert to [0, 1] range
    transforms.Normalize(mean, std)       # Standardization
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

**Your answer:**

In [50]:
# Call the dataset Loader
trainset = CIFAR10(root=DATAROOT, train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=1)
valset = CIFAR10(root=DATAROOT, train=False, download=True, transform=transform_val)
valloader = torch.utils.data.DataLoader(valset, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=1)

Using downloaded and verified file: ./data/cifar10_trainval.tar.gz
Extracting ./data/cifar10_trainval.tar.gz to ./data
Files already downloaded and verified
Using downloaded and verified file: ./data/cifar10_trainval.tar.gz
Extracting ./data/cifar10_trainval.tar.gz to ./data
Files already downloaded and verified


In [51]:
# FLAG for loading the pretrained model
TRAIN_FROM_SCRATCH = True
# Code for loading checkpoint and recover epoch id.
CKPT_PATH = "./saved_model/model.h5"
def get_checkpoint(ckpt_path):
    try:
        ckpt = torch.load(ckpt_path)
    except Exception as e:
        print(e)
        return None
    return ckpt

ckpt = get_checkpoint(CKPT_PATH)
if ckpt is None or TRAIN_FROM_SCRATCH:
    if not TRAIN_FROM_SCRATCH:
        print("Checkpoint not found.")
    print("Training from scratch ...")
    start_epoch = 0
    current_learning_rate = INITIAL_LR
else:
    print("Successfully loaded checkpoint: %s" %CKPT_PATH)
    net.load_state_dict(ckpt['net'])
    start_epoch = ckpt['epoch'] + 1
    current_learning_rate = ckpt['lr']
    print("Starting from epoch %d " %start_epoch)

print("Starting from learning rate %f:" %current_learning_rate)

Training from scratch ...
Starting from learning rate 0.010000:


In [52]:
# Specify the device for computation
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = LeNet5(use_bn=True)
net = net.to(device)
if device =='cuda':
    print("Train on GPU...")
else:
    print("Train on CPU...")

Train on GPU...


In [ ]:
"""
Loss Function and Optimizer Initialization

Objective: Set up the learning objective and optimization algorithm.
We use Cross-Entropy Loss, which is standard for multi-class classification.
For optimization, we use Stochastic Gradient Descent (SGD) with momentum
and weight decay (L2 regularization) to update the model parameters.
"""
# Create loss function and specify regularization
criterion = nn.CrossEntropyLoss()
# Add optimizer
optimizer = optim.SGD(
    net.parameters(),
    lr=INITIAL_LR,
    momentum=MOMENTUM,
    weight_decay=REG # L2
)

In [ ]:
"""
Main Training and Validation Loop

Objective: Train the LeNet-5 model on the CIFAR-10 dataset.
We iterate through the dataset in batches, performing forward passes,
calculating loss, and backpropagating gradients. After each epoch, we
evaluate the model on the validation set to track generalization performance
and save the best performing model checkpoint.
"""
history = {
    "epoch": [],
    "train_acc": [],
    "val_acc": [],
}

global_step = 0
best_val_acc = 0

for i in range(start_epoch, EPOCHS):
    print(datetime.datetime.now())
    # Switch to train mode
    net.train()
    print("Epoch %d:" %i)

    total_examples = 0
    correct_examples = 0

    train_loss = 0
    train_acc = 0
    # Train the training dataset for 1 epoch.
    print(len(trainloader))
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        # Copy inputs to device
        inputs = inputs.to(device)
        targets = targets.to(device)
        # Zero the gradient
        optimizer.zero_grad()
        # Generate output
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        # Now backward loss
        loss.backward()
        # Apply gradient
        optimizer.step()
        # Calculate predicted labels
        _, predicted = outputs.max(1)
        # Calculate accuracy
        total_examples += targets.size(0)
        correct_examples += predicted.eq(targets).sum().item()

        train_loss += loss.item()

        global_step += 1
        if global_step % 100 == 0:
            avg_loss = train_loss / (batch_idx + 1)
        pass
    avg_acc = correct_examples / total_examples
    print("Training loss: %.4f, Training accuracy: %.4f" %(avg_loss, avg_acc))
    print(datetime.datetime.now())
    # Validate on the validation dataset
    print("Validation...")
    total_examples = 0
    correct_examples = 0

    net.eval()

    val_loss = 0
    val_acc = 0
    # Disable gradient during validation
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(valloader):
            # Copy inputs to device
            inputs = inputs.to(device)
            targets = targets.to(device)
            # Zero the gradient
            optimizer.zero_grad()
            # Generate output from the DNN.
            outputs = net(inputs)
            loss = criterion(outputs, targets)
            # Calculate predicted labels
            _, predicted = outputs.max(1)
            # Calculate accuracy
            total_examples += targets.size(0)
            correct_examples += predicted.eq(targets).sum().item()
            val_loss += loss.item()

    avg_loss_val = val_loss / len(valloader)
    avg_acc_val = correct_examples / total_examples
    print("Validation loss: %.4f, Validation accuracy: %.4f" % (avg_loss_val, avg_acc_val))


    history["epoch"].append(i)
    history["train_acc"].append(avg_acc)
    history["val_acc"].append(avg_acc_val)

    """
    Part 4(b) & 4(c): Learning Rate Decay Schedule
    
    Objective: Dynamically adjust the learning rate during training.
    Gradually decaying the learning rate allows the model to take smaller
    optimization steps as it approaches a minimum, preventing it from
    overshooting and improving final validation accuracy.
    """
    DECAY_EPOCHS = 2
    DECAY = 0.98
    if i % DECAY_EPOCHS == 0 and i != 0:
        current_learning_rate = current_learning_rate * DECAY
        for param_group in optimizer.param_groups:
            # Assign the learning rate parameter
            param_group['lr'] = current_learning_rate

        print("Current learning rate has decayed to %f" %current_learning_rate)

    # Save for checkpoint
    if avg_acc_val > best_val_acc:
        best_val_acc = avg_acc_val
        if not os.path.exists(CHECKPOINT_PATH):
            os.makedirs(CHECKPOINT_PATH)
        print("Saving ...")
        state = {'net': net.state_dict(),
                 'epoch': i,
                 'lr': current_learning_rate}
        torch.save(state, os.path.join(CHECKPOINT_PATH, 'model.h5'))

print("Optimization finished.")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import copy

"""Visualization and Analysis"""

def plot_learning_curve(history, title="Training vs Validation Accuracy"):
    """
    Plot training and validation accuracy over training epochs.

    :param history: Dictionary containing training history with keys "epoch", "train_acc", and "val_acc".
    :type history: dict
    :param title: Title of the plot.
    :type title: str
    """
    plt.figure()
    plt.plot(history["epoch"], history["train_acc"], label="Training Accuracy")
    plt.plot(history["epoch"], history["val_acc"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.show()

import pandas as pd

def history_to_dataframe(history):
    """
    Convert the history dictionary into a pandas DataFrame.

    :param history: Dictionary containing training history with keys "epoch", "train_acc", and "val_acc".
    :type history: dict
    :return: DataFrame with columns for epoch, training accuracy, and validation accuracy.
    :rtype: pandas.DataFrame
    """
    df = pd.DataFrame({
        "epoch": history["epoch"],
        "train_acc": history["train_acc"],
        "val_acc": history["val_acc"]
    })
    return df